In [11]:
import os
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from lightgbm import LGBMRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import train_test_split
from math import sqrt

import warnings
warnings.filterwarnings('ignore')

In [3]:
data_ace_24 = pd.read_csv("../data/Ace_погружение_24часа.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')
data_discover_24 = pd.read_csv("../data/Discover_погружение_24часа.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')
data_ace_af = pd.read_csv("../data/Ace_погружение_AF.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')
data_discover_af = pd.read_csv("../data/Discover_погружение_AF.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')

In [4]:
class DANNRegressor:
    """
    Domain-Adversarial Neutral Network для адаптации discover -> ace
    fit(X_src, X_tgt), predict(X_src_test) -> X_tgt_hat
    """

    def __init__(self,
                 hidden_layer_size=64,
                 learning_rate=1e-3,
                 lambda_adapt=1.0,
                 maxiter=20,
                 adversarial_representation=True,
                 seed=42,
                 verbose=False):
        self.hidden_layer_size = hidden_layer_size
        self.learning_rate = learning_rate
        self.lambda_adapt = lambda_adapt if lambda_adapt not in (None, False) else 0.0
        self.maxiter = maxiter
        self.adversarial_representation = adversarial_representation
        self.seed = seed
        self.verbose = verbose

    def sigmoid(self, z):
        return 1.0 / (1.0 + np.exp(-z))

    def random_init(self, l_in, l_out):
        """ l_in - размер входных нейронов слоя, l_out - выходных """
        eps = sqrt(6.0 / (l_in + l_out))  # xavier initialization
        return eps * (2 * np.random.rand(l_out, l_in) - 1.0)

    def fit(self, X_src, X_tgt):
        """
        X_src: (N, D) данные discover
        X_tgt: (N, D) данные ace
        """
        N, D = X_src.shape
        K = D  # такая же размерность

        np.random.seed(self.seed)

        # вход -> скрытый слой
        W = self.random_init(D, self.hidden_layer_size)   # (H, D)
        b = np.zeros(self.hidden_layer_size)              # (H,)

        # скрытый слой -> выход (регрессия discover -> ace)
        V = self.random_init(self.hidden_layer_size, K)   # (K, H)
        c = np.zeros(K)                                   # (K,)

        # доменный классификатор по скрытому слою
        # random init, чтобы адаптация не спала в начале
        U = np.zeros(self.hidden_layer_size)              # (H,)
        d = 0.0                                           # скаляр

        for epoch in range(self.maxiter):
            for i in range(N):
                x_s = X_src[i, :]
                y_s = X_tgt[i, :]

                # регрессионная часть source -> target
                hidden_s = self.sigmoid(W @ x_s + b)   # (H,)
                y_pred = V @ hidden_s + c              # (K,)

                # задаем функцию потерь mse: L = 1/2 ||y_pred - y_s||^2
                err = y_pred - y_s                     # dL/dy_pred (K,)

                # градиенты по выходному слою (в исходном коде здесь softmax)
                delta_c = err                          # dL/dc
                delta_V = np.outer(err, hidden_s)      # dL/dV

                # dL/dh = V^T err, затем через сигмоид
                delta_hidden = V.T @ err               # dL/dh
                delta_hidden *= hidden_s * (1.0 - hidden_s)  # dL/dz
                delta_b = delta_hidden                 # dL/db
                delta_W = np.outer(delta_hidden, x_s)  # dL/dW

                # доменная часть
                if self.lambda_adapt != 0.0:
                    g_s = self.sigmoid(U @ hidden_s + d)   # доменная вероятность

                    # delta_d = λ (1 - g_s)
                    # delta_U = delta_d * hidden_s
                    delta_d = self.lambda_adapt * (1.0 - g_s)
                    delta_U = delta_d * hidden_s

                    if self.adversarial_representation:
                        # tmp = delta_d * U * h * (1 - h)
                        tmp = delta_d * U * hidden_s * (1.0 - hidden_s)
                        delta_b += tmp
                        delta_W += np.outer(tmp, x_s)

                    # Добавляем регуляризатор от другого домена (target)
                    j = np.random.randint(N)
                    x_t = X_tgt[j, :]
                    hidden_t = self.sigmoid(W @ x_t + b)
                    g_t = self.sigmoid(U @ hidden_t + d)

                    delta_d -= self.lambda_adapt * g_t
                    delta_U -= self.lambda_adapt * g_t * hidden_t

                    if self.adversarial_representation:
                        tmp = -self.lambda_adapt * g_t * U * hidden_t * (1.0 - hidden_t)
                        delta_b += tmp
                        delta_W += np.outer(tmp, x_t)
                else:
                    delta_U = 0.0
                    delta_d = 0.0

                lr = self.learning_rate

                # минимизируем регрессионный loss + доменный регуляризатор
                W -= lr * delta_W
                b -= lr * delta_b
                V -= lr * delta_V
                c -= lr * delta_c

                # максимизирует расхождение
                U += lr * delta_U
                d += lr * delta_d

            if self.verbose:
                hidden_all = self.sigmoid(W @ X_src.T + b[:, None])     # (H, N)
                y_hat = (V @ hidden_all + c[:, None]).T                 # (N, K)
                mse = float(np.mean((y_hat - X_tgt) ** 2))
                print(f"Epoch {epoch+1}/{self.maxiter}, MSE source={mse:.4f}")

        self.W, self.V, self.b, self.c, self.U, self.d = W, V, b, c, U, d
        return self

    def _hidden(self, X):
        return self.sigmoid(self.W @ X.T + self.b[:, None])

    def predict(self, X_src):
        h = self._hidden(X_src)               # (H, N)
        Y_hat = self.V @ h + self.c[:, None]  # (K, N)
        return Y_hat.T                        # (N, K)

    def predict_domain(self, X):
        """0 - source, 1 - target (по скрытому слою)"""
        h = self._hidden(X)                   # (H, N)
        g = self.sigmoid(self.U @ h + self.d) # (N,)
        return (g < 0.5).astype(int)  #  <0.5 -> один домен, >0.5 -> другой


In [22]:
def adapt_ace_to_discover(ace_df, disc_df, future_lags, model, split_date="2021-01-01"):
    """
    Адаптация ace -> discover

    1. K обучается только на пересечении тренировочных данных ace и discover
    2. Ace до появления discover адаптируется в домен discover
    3. Возвращаются адаптированный ace_old_train и disc_train, disc_test
    """

    ace = ace_df.sort_index()
    disc = disc_df.sort_index()
    split_date = pd.Timestamp(split_date)

    ace_train = ace.loc[:split_date]
    disc_train = disc.loc[:split_date]
    disc_test = disc.loc[split_date:]

    overlap_idx = ace_train.index.intersection(disc_train.index)

    ace_overlap = ace_train.loc[overlap_idx]
    disc_overlap = disc_train.loc[overlap_idx]

    feature_cols = [c for c in ace.columns if c not in future_lags]

    # Масштабирование
    sc_ace = StandardScaler().fit(ace_overlap[feature_cols])
    sc_disc = StandardScaler().fit(disc_overlap[feature_cols])

    X_ace = sc_ace.transform(ace_overlap[feature_cols])
    X_disc = sc_disc.transform(disc_overlap[feature_cols])
   
    K = model
    K.fit(X_ace, X_disc)

    # ace до появления discover
    ace_old = ace.loc[:disc.index.min()]
    ace_old = ace_old[feature_cols]

    X_ace_old = sc_ace.transform(ace_old)
    X_ace_old_adapted = K.predict(X_ace_old)
    X_ace_old_adapted = sc_disc.inverse_transform(X_ace_old_adapted)

    ace_old_adapted = pd.DataFrame(X_ace_old_adapted, index=ace_old.index, columns=feature_cols)

    # Целевые переменные возвращаются обратно неизменёнными
    for col in future_lags:
        ace_old_adapted[col] = ace_df.loc[ace_old.index, col]

    return ace_old_adapted, disc_train, disc_test, K, sc_ace, sc_disc

In [12]:
def build_models(random_state=42):
    models = {}
    models['Linear'] = Pipeline([
        ("scaler", StandardScaler()),
        ("lin", LinearRegression())
    ])
    
    models['Ridge'] = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(
            alpha=1.0,
            solver='auto',
            random_state=random_state))
    ])

    models['Lasso'] = Pipeline([
        ("scaler", StandardScaler()),
        ("lasso", Lasso(
            alpha=0.0005,
            tol=0.01,
            max_iter=500, 
            random_state=random_state))
    ])
    
    models['LGBM'] = Pipeline([
        ("boost", LGBMRegressor(
            n_estimators=1500,
            max_depth=3,
            early_stopping_rounds=50,
            random_state=random_state,
            verbose=-1))
    ])
    
    models['MLP'] = Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPRegressor(
            hidden_layer_sizes=(256, 256, 128),
            solver='adam',
            alpha=1e-4,
            learning_rate_init=1e-3,
            early_stopping=True,
            validation_fraction=0.1,
            max_iter=1500,
            random_state=random_state))
    ])
    return models
    
models = build_models()

def evaluate_M_D(ace_old_adapted, disc_train, disc_test, split_date, target, future_lags, adaptation_method='linreg', delays='24h', results_list=None):
    """
    Обучение M_D:
    - train = адаптированный старый ace + discover_train
    - test = discover_test
    """
    train_adapt = pd.concat([ace_old_adapted, disc_train], axis=0).sort_index()

    feature_cols = [c for c in train_adapt.columns if c not in future_lags]

    X_train = train_adapt[feature_cols].values
    y_train = train_adapt[target].values

    X_test = disc_test[feature_cols].values
    y_test = disc_test[target].values
    
    results = {}
    results[target] = {}

    for name, model in models.items():
        if name == 'LGBM':
            # Для бустинга выделяем валидационный набор
            X_tr, X_val, y_tr, y_val = train_test_split(
                X_train, y_train, test_size=0.1, random_state=42)

            model.fit(X_tr, y_tr,
                boost__eval_set=[(X_val, y_val)],
                boost__eval_metric='l2')
        else:
            model.fit(X_train, y_train)
            
        y_pred = model.predict(X_test)
        
        if results_list is None:
            results_list = []
            
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        results_list.append({
            'target': target,
            'delays': delays,
            'adaptation_method': adaptation_method,
            'forecast_model': name,
            'RMSE': rmse,
            'MAE': mae,
            'R2': r2
        })

        print(f"{name}: rmse={rmse:.4f}, mae={mae:.4f}, r2={r2:.4f}")

    return results_list

In [29]:
def evaluate_M_A_check_params(ace_df, disc_test_adapted_df, split_date, target, future_lags, adaptation_method='linreg', delays='24h'):

    split_date = pd.Timestamp(split_date)

    ace_train = ace_df.loc[:split_date]

    feature_cols = [c for c in ace_train.columns if c not in future_lags]

    X_train = ace_train[feature_cols].values
    y_train = ace_train[target].values

    X_test = disc_test_adapted_df[feature_cols].values
    y_test = disc_test_adapted_df[target].values

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("lin", LinearRegression())
    ])
            
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
        
    # rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    # mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    return r2

In [17]:
# 0. Копирование загруженных данных в отдельные переменные
data_ace_24_copy = data_ace_24.copy()
data_discover_24_copy = data_discover_24.copy()
data_ace_af_copy = data_ace_af.copy()
data_discover_af_copy = data_discover_af.copy()

# 1. Задание переменных для адаптации
split_date = "2022-01-01"
future_lags = [f'Dst_plus{i}' for i in range(1, 25)] # все сдвиги во времени вперед (задаются при обработке данных)
targets = ['Dst_plus1', 'Dst_plus2', 'Dst_plus3', 'Dst_plus6'] # таргеты, на которые делаем прогнозы

In [49]:
target = 'Dst_plus1'

# модель полегче для более быстрого теста
hidden = 512
lr = 4e-4
lam = 3.0
# maxiter = 10


# lam_params = [0.8, 1.5, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 20, 50]
# lr_params = [1e-6, 1e-5, 1e-4, 1e-3, 1e-2] 
# lr_params = [3e-4, 4e-4, 5e-4, 6e-4, 7e-4, 8e-4, 9e-4, 1e-3, 2e-3, 3e-3, 4e-3, 5e-3, 7e-3] # 4e-4
# hidden_params = [[64, 128, 256, 512, 1024] 
iter_params = [10, 20, 30, 40]

# for hidden in hidden_params:
    # print(hidden)
# for lr in lr_params:
    # print(lr)
# for lam in lam_params:
#     print(lam)
for maxiter in iter_params:
    print(maxiter)
    
    model = DANNRegressor(
        hidden_layer_size=hidden,
        learning_rate=lr,
        lambda_adapt=lam,
        maxiter=maxiter,
        verbose=False
    )

    disc_test_adapted = adapt_ace_to_discover(
        data_ace_24_copy,
        data_discover_24_copy,
        future_lags,
        model,
        split_date
    )

    disc_test_adapted_df = disc_test_adapted[0]

    r2 = evaluate_M_A_check_params(
        data_ace_24_copy,
        disc_test_adapted_df,
        split_date,
        target,
        future_lags
    )
    print(r2)

10
0.8722805047110274
20
0.8842538097047185
30
0.8887528532901698
40
0.8970000971212838


In [65]:
model = DANNRegressor(
    hidden_layer_size=256,
    learning_rate=4e-4,
    lambda_adapt=2,
    maxiter=30,
    verbose=True
)

In [66]:
ace_adapted_dann_a_to_d_24, disc_train_24, disc_test_24, K_lin_24, sd_lin_24, sa_lin_24 = adapt_ace_to_discover(
    data_ace_24_copy,
    data_discover_24_copy,
    future_lags,
    model,
    split_date
)

Epoch 1/30, MSE source=0.2930
Epoch 2/30, MSE source=0.2850
Epoch 3/30, MSE source=0.2883
Epoch 4/30, MSE source=0.2927
Epoch 5/30, MSE source=0.2982
Epoch 6/30, MSE source=0.3022
Epoch 7/30, MSE source=0.3059
Epoch 8/30, MSE source=0.3088
Epoch 9/30, MSE source=0.3124
Epoch 10/30, MSE source=0.3151
Epoch 11/30, MSE source=0.3174
Epoch 12/30, MSE source=0.3205
Epoch 13/30, MSE source=0.3240
Epoch 14/30, MSE source=0.3286
Epoch 15/30, MSE source=0.3296
Epoch 16/30, MSE source=0.3277
Epoch 17/30, MSE source=0.3285
Epoch 18/30, MSE source=0.3282
Epoch 19/30, MSE source=0.3290
Epoch 20/30, MSE source=0.3276
Epoch 21/30, MSE source=0.3268
Epoch 22/30, MSE source=0.3272
Epoch 23/30, MSE source=0.3288
Epoch 24/30, MSE source=0.3309
Epoch 25/30, MSE source=0.3331
Epoch 26/30, MSE source=0.3341
Epoch 27/30, MSE source=0.3347
Epoch 28/30, MSE source=0.3337
Epoch 29/30, MSE source=0.3356
Epoch 30/30, MSE source=0.3365


In [67]:
ace_adapted_dann_a_to_d_af, disc_train_af, disc_test_af, K_lin_af, sd_lin_af, sa_lin_af = adapt_ace_to_discover(
    data_ace_af_copy,
    data_discover_af_copy,
    future_lags,
    model,
    split_date
)

Epoch 1/30, MSE source=0.2586
Epoch 2/30, MSE source=0.2441
Epoch 3/30, MSE source=0.2427
Epoch 4/30, MSE source=0.2434
Epoch 5/30, MSE source=0.2444
Epoch 6/30, MSE source=0.2459
Epoch 7/30, MSE source=0.2474
Epoch 8/30, MSE source=0.2499
Epoch 9/30, MSE source=0.2531
Epoch 10/30, MSE source=0.2560
Epoch 11/30, MSE source=0.2587
Epoch 12/30, MSE source=0.2618
Epoch 13/30, MSE source=0.2640
Epoch 14/30, MSE source=0.2659
Epoch 15/30, MSE source=0.2701
Epoch 16/30, MSE source=0.2723
Epoch 17/30, MSE source=0.2743
Epoch 18/30, MSE source=0.2774
Epoch 19/30, MSE source=0.2776
Epoch 20/30, MSE source=0.2785
Epoch 21/30, MSE source=0.2797
Epoch 22/30, MSE source=0.2802
Epoch 23/30, MSE source=0.2821
Epoch 24/30, MSE source=0.2845
Epoch 25/30, MSE source=0.2861
Epoch 26/30, MSE source=0.2884
Epoch 27/30, MSE source=0.2889
Epoch 28/30, MSE source=0.2898
Epoch 29/30, MSE source=0.2906
Epoch 30/30, MSE source=0.2933


In [74]:
results_list = []

print(f"\n==== Depth - 24h ====")

print(f"\n==== Adaptation - ace-to-discover DANN ====")
for target_col in targets:
    print(f"\n==== Forecast of {target_col.upper()} ====")
    res = evaluate_M_D(ace_adapted_dann_a_to_d_24,
                       disc_train_24,
                       disc_test_24,
                       split_date,
                       target_col,
                       future_lags,
                       adaptation_method='ace-disc-dann',
                       delays='24h',
                       results_list=results_list)


==== Depth - 24h ====

==== Adaptation - ace-to-discover DANN ====

==== Forecast of DST_PLUS1 ====
Linear: rmse=4.7156, mae=3.3147, r2=0.9360
Ridge: rmse=4.7149, mae=3.3142, r2=0.9360
Lasso: rmse=4.6653, mae=3.2712, r2=0.9374
LGBM: rmse=5.7728, mae=3.0504, r2=0.9041
MLP: rmse=5.7545, mae=3.0703, r2=0.9047

==== Forecast of DST_PLUS2 ====
Linear: rmse=6.2425, mae=4.4833, r2=0.8878
Ridge: rmse=6.2419, mae=4.4828, r2=0.8878
Lasso: rmse=6.2063, mae=4.4533, r2=0.8891
LGBM: rmse=6.9315, mae=4.1860, r2=0.8616
MLP: rmse=8.5320, mae=4.9674, r2=0.7904

==== Forecast of DST_PLUS3 ====
Linear: rmse=7.5521, mae=5.4223, r2=0.8357
Ridge: rmse=7.5516, mae=5.4220, r2=0.8357
Lasso: rmse=7.5261, mae=5.3968, r2=0.8368
LGBM: rmse=8.3511, mae=5.1378, r2=0.7991
MLP: rmse=9.9997, mae=6.0528, r2=0.7119

==== Forecast of DST_PLUS6 ====
Linear: rmse=10.5556, mae=7.3957, r2=0.6786
Ridge: rmse=10.5553, mae=7.3954, r2=0.6786
Lasso: rmse=10.5495, mae=7.3843, r2=0.6790
LGBM: rmse=11.8139, mae=7.4211, r2=0.5974
MLP:

In [69]:
print(f"\n==== Depth - autocorrelation function ====")

print(f"\n==== Adaptation - ace-to-discover DANN ====")
for target_col in targets:
    print(f"\n==== Forecast of {target_col.upper()} ====")
    res = evaluate_M_D(ace_adapted_dann_a_to_d_af,
                       disc_train_af,
                       disc_test_af,
                       split_date,
                       target_col,
                       future_lags,
                       adaptation_method='ace-disc-dann',
                       delays='auto_func',
                       results_list=results_list)


==== Depth - autocorrelation function ====

==== Adaptation - ace-to-discover DANN ====

==== Forecast of DST_PLUS1 ====
Linear: rmse=4.9221, mae=3.4627, r2=0.9299
Ridge: rmse=4.9212, mae=3.4620, r2=0.9299
Lasso: rmse=4.8769, mae=3.4199, r2=0.9312
LGBM: rmse=6.8985, mae=3.2222, r2=0.8623
MLP: rmse=5.7167, mae=3.1000, r2=0.9055

==== Forecast of DST_PLUS2 ====
Linear: rmse=6.4449, mae=4.6680, r2=0.8798
Ridge: rmse=6.4441, mae=4.6674, r2=0.8799
Lasso: rmse=6.4017, mae=4.6303, r2=0.8814
LGBM: rmse=7.9170, mae=4.3986, r2=0.8187
MLP: rmse=7.8299, mae=4.8514, r2=0.8226

==== Forecast of DST_PLUS3 ====
Linear: rmse=7.7214, mae=5.6080, r2=0.8275
Ridge: rmse=7.7208, mae=5.6075, r2=0.8275
Lasso: rmse=7.6819, mae=5.5719, r2=0.8293
LGBM: rmse=9.1327, mae=5.3748, r2=0.7587
MLP: rmse=9.7871, mae=6.2459, r2=0.7229

==== Forecast of DST_PLUS6 ====
Linear: rmse=10.6191, mae=7.5693, r2=0.6738
Ridge: rmse=10.6188, mae=7.5690, r2=0.6738
Lasso: rmse=10.5954, mae=7.5479, r2=0.6752
LGBM: rmse=12.0393, mae=7

In [75]:
results_df = pd.DataFrame(results_list)

In [76]:
results_df

,target,delays,adaptation_method,forecast_model,RMSE,MAE,R2
0,Dst_plus1,24h,ace-disc-dann,Linear,4.715554,3.314701,0.935994
1,Dst_plus1,24h,ace-disc-dann,Ridge,4.714868,3.314154,0.936013
2,Dst_plus1,24h,ace-disc-dann,Lasso,4.665323,3.271233,0.937351
3,Dst_plus1,24h,ace-disc-dann,LGBM,5.772754,3.050424,0.904078
4,Dst_plus1,24h,ace-disc-dann,MLP,5.754535,3.070315,0.904682
5,Dst_plus2,24h,ace-disc-dann,Linear,6.242474,4.483267,0.887789
6,Dst_plus2,24h,ace-disc-dann,Ridge,6.241932,4.482849,0.887808
7,Dst_plus2,24h,ace-disc-dann,Lasso,6.206339,4.453311,0.889084
8,Dst_plus2,24h,ace-disc-dann,LGBM,6.931517,4.185996,0.861650
9,Dst_plus2,24h,ace-disc-dann,MLP,8.531961,4.967398,0.790386


In [77]:
results_df.to_excel("../results/models_adaptation_ace_to_discover_dann_2022-01-01.xlsx", index=False)